# Comparatif TTS v4 : temps d'inférence, comparaison équitable
**Ce qu'on mesure :** le **temps de calcul** pour générer l'audio d'une phrase (pas la durée de l'audio). Chargement du modèle exclu.

**Mode d'emploi :** *Exécution → Modifier le type d'exécution → GPU T4*, puis *Exécution → Tout exécuter*. Ne ferme pas l'onglet. Compte **1 h 30 à 2 h**. À la fin, 3 fichiers se téléchargent : `resultats_inference.csv`, `bench_audio.zip`, `bench_logs.zip`.

La phrase testée : *« Bonjour, je vous invite à découvrir notre nouvelle application qui vous aidera à apprendre plus facilement chaque jour. »*

## Les règles d'équité (identiques pour tous les modèles)
1. **Même phrase, même GPU, même session Colab** : on lance tout en une fois.
2. **Un modèle à la fois, chacun dans son propre processus et son propre environnement Python** : aucun modèle ne ralentit un autre, aucune bibliothèque n'interfère.
3. **Chronomètre précis** (`time.perf_counter`) **et synchronisé avec le GPU** : on attend que le GPU ait vraiment fini avant d'arrêter le chrono.
4. **Ce qui est chronométré :** de l'envoi du texte au modèle jusqu'à l'audio prêt en mémoire. **Exclu :** le chargement du modèle, la préparation unique de la voix, l'écriture du fichier final.
5. **Échauffement non chronométré avec la vraie phrase**, puis **5 essais chronométrés**, avec des **graines aléatoires fixées** (essai 1 = 1001, etc.) : relancer le notebook redonne les mêmes audios.
6. **Même voix de référence (« Claire ») pour tous les modèles de clonage**, et **préparée une seule fois avant le chrono** quand le modèle le permet. Quand ce n'est pas possible (F5-TTS, KokoClone), c'est écrit dans la remarque.
7. **Réglages par défaut des auteurs.** Quand un modèle est testé en demi-précision (plus rapide), c'est écrit dans son nom ; s'il devient instable en demi-précision, il repasse automatiquement en float32 et c'est écrit dans la remarque.
8. **Contrôle automatique :** un audio vide ou cassé est une erreur, et **Whisper vérifie que chaque audio dit bien la phrase**. Un essai rapide mais faux (phrase coupée) est signalé dans la remarque.

## Résultat final
**Un tableau par catégorie** (clonage, design, voix prédéfinie), car **la comparaison se fait par catégorie** : les modèles d'une même catégorie reçoivent la même entrée (même enregistrement « Claire », ou une description écrite, ou une voix intégrée). Pour chaque modèle : **rang dans sa catégorie, temps des 5 essais, moyenne, remarque**. Le fichier `resultats_inference.csv` contient les mêmes colonnes, trié par catégorie. Les détails techniques (mémoire GPU, durée de l'audio, transcriptions) sont rangés dans `bench_logs.zip`.

*Option :* pour cloner une **vraie voix humaine** au lieu de « Claire » (générée par Kokoro), dépose un fichier `ref_fr.wav` (5 à 15 s) dans `/content/bench/` avant de lancer, mets `USE_OWN_REF = True` à l'étape 1 et écris sa transcription exacte dans `REF_TEXT`.

In [ ]:
# ===== ÉTAPE 1 : installation de base + fichiers communs (≈ 2 min) =====
import os, subprocess
subprocess.run("apt-get -qq update && apt-get -qq install -y espeak-ng ffmpeg git-lfs libsndfile1 sox > /dev/null 2>&1", shell=True)
subprocess.run("pip -q install uv huggingface_hub soundfile pandas", shell=True)
for d in ["/content/bench/logs", "/content/bench/out", "/content/envs", "/content/uvcache"]:
    os.makedirs(d, exist_ok=True)

N_RUNS = 5              # nombre de mesures chronométrées par modèle
USE_OWN_REF = False     # True = tu as déposé ta propre voix dans /content/bench/ref_fr.wav

TEXT = "Bonjour, je vous invite à découvrir notre nouvelle application qui vous aidera à apprendre plus facilement chaque jour."
REF_TEXT = "Bonjour, je m'appelle Claire et je suis ravie de vous présenter ce projet."   # transcription EXACTE de la voix de référence
open("/content/bench/text.txt", "w", encoding="utf-8").write(TEXT)
open("/content/bench/ref_text.txt", "w", encoding="utf-8").write(REF_TEXT)
if not USE_OWN_REF and os.path.exists("/content/bench/ref_fr.wav"):
    os.remove("/content/bench/ref_fr.wav")      # sera régénérée par Kokoro
if USE_OWN_REF:
    assert os.path.exists("/content/bench/ref_fr.wav"), "Dépose d'abord ta voix dans /content/bench/ref_fr.wav"

COMMON = r"""
import os, sys, time, json, statistics, random, gc
os.environ.setdefault('MPLBACKEND', 'Agg')
import numpy as np, soundfile as sf
TEXT = open('/content/bench/text.txt', encoding='utf-8').read().strip()
REF_TEXT = open('/content/bench/ref_text.txt', encoding='utf-8').read().strip()
REF = '/content/bench/ref_fr.wav'
NAME = os.environ['BENCH_NAME']
N_RUNS = int(os.environ.get('N_RUNS', '5'))
HALF = os.environ.get('HALF', '0') == '1'
try:
    import torch
    DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
except Exception:
    torch = None
    DEV = 'cpu'

def half_dtype():
    # bfloat16 seulement sur les GPU récents (L4, A100...) ; sur T4 on prend float16
    if DEV == 'cuda' and torch.cuda.get_device_capability()[0] >= 8:
        return torch.bfloat16
    return torch.float16

def dtype_name(dt):
    return str(dt).replace('torch.', '')

ORT = []   # fournisseurs ONNX réellement utilisés
try:
    import onnxruntime as _ort
    _orig = _ort.InferenceSession.__init__
    def _patched(self, *a, **k):
        _orig(self, *a, **k)
        try: ORT.append(self.get_providers()[0])
        except Exception: pass
    _ort.InferenceSession.__init__ = _patched
except Exception:
    pass

def sync():
    try:
        if torch is not None and torch.cuda.is_available(): torch.cuda.synchronize()
    except Exception:
        pass

def seed(s):
    random.seed(s); np.random.seed(s)
    try:
        torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except Exception:
        pass

def to_np(a):
    if hasattr(a, 'detach'): a = a.detach().cpu().float().numpy()
    return np.asarray(a, dtype=np.float32).squeeze()

def check_audio(a, sr, label):
    if a.size == 0 or not np.all(np.isfinite(a)):
        raise RuntimeError('audio invalide (%s) : vide ou NaN' % label)
    if float(np.sqrt(np.mean(a ** 2))) < 1e-4 or len(a) / sr < 0.5:
        raise RuntimeError('audio invalide (%s) : silence ou < 0,5 s' % label)

def materialize(a, sr):
    if callable(a): a, sr = a()
    return to_np(a), sr

def load_with_half_fallback(load, make_synth):
    # Demi-précision si demandée ; si l'audio de test est cassé, on recharge en float32.
    if not (HALF and DEV == 'cuda'):
        m = load(torch.float32 if torch is not None else None)
        return m, make_synth(m), 'float32'
    dt = half_dtype()
    try:
        m = load(dt); s = make_synth(m)
        a, sr = materialize(*s('Bonjour, ceci est un test.'))
        check_audio(a, sr, 'test demi-précision')
        return m, s, dtype_name(dt)
    except Exception as e:
        print('Demi-précision instable, retour en float32 :', e)
        m = None; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        m = load(torch.float32)
        return m, make_synth(m), 'float32 (demi-précision instable, repli automatique)'

def run(load_s, synth, info=''):
    seed(0)
    w, wsr = materialize(*synth(TEXT))          # échauffement avec la VRAIE phrase (non chronométré)
    check_audio(w, wsr, 'échauffement')
    times, durs = [], []
    for i in range(N_RUNS):
        seed(1001 + i)
        sync(); t = time.perf_counter()
        a, sr = synth(TEXT)                      # <<< début → fin de la génération
        sync(); dt = time.perf_counter() - t     # le chrono s'arrête quand le GPU a fini
        a, sr = materialize(a, sr)               # (lecture d'un éventuel fichier temporaire : hors chrono)
        check_audio(a, sr, 'essai %d' % (i + 1))
        sf.write('/content/bench/out/%s_r%d.wav' % (NAME, i + 1), a, sr)
        times.append(dt); durs.append(len(a) / sr)
        if dt > 120: break                       # modèle très lent : on s'arrête là
    sf.write('/content/bench/out/%s.wav' % NAME, a, sr)
    vram = 0.0
    try:
        if torch.cuda.is_available(): vram = torch.cuda.max_memory_allocated() / 1e9
    except Exception:
        pass
    print('RESULT ' + json.dumps(dict(load_s=round(load_s, 2), times=[round(x, 3) for x in times],
        durs=[round(x, 2) for x in durs], mean_s=round(statistics.mean(times), 3),
        std_s=round(statistics.pstdev(times), 3), n=len(times),
        device=DEV, ort=sorted(set(ORT)), vram_gb=round(vram, 2), info=info)))
"""
open("/content/bench/common.py", "w").write(COMMON)

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
GPU_NAME = gpu if gpu else "CPU"
open("/content/bench/logs/gpu.txt", "w").write(GPU_NAME)
print("Matériel détecté :", GPU_NAME)
assert gpu, "❌ Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4, puis relance."
print("Étape 1 terminée ✅")

In [ ]:
# ===== ÉTAPE 2 : les scripts de chaque modèle (rien à modifier) =====
H = "import sys; sys.path.insert(0, '/content/bench')\nfrom common import *\n"
RUNNERS = {}

# ---------- Voix prédéfinies ----------
RUNNERS["kokoro"] = H + r"""
t = time.perf_counter()
from kokoro import KPipeline
pipe = KPipeline(lang_code='f', repo_id='hexgrad/Kokoro-82M')
load = time.perf_counter() - t
def synth(txt):
    return np.concatenate([to_np(a) for _, _, a in pipe(txt, voice='ff_siwis')]), 24000
if not os.path.exists(REF):                     # voix de référence « Claire » pour tous les modèles de clonage
    seed(0); sf.write(REF, synth(REF_TEXT)[0], 24000)
run(load, synth, info='voix ff_siwis')
"""

RUNNERS["melo"] = H + r"""
t = time.perf_counter()
from melo.api import TTS
model = TTS(language='FR', device=DEV)
spk = model.hps.data.spk2id['FR']
load = time.perf_counter() - t
def synth(txt):
    return model.tts_to_file(txt, spk, None, speed=1.0), model.hps.data.sampling_rate
run(load, synth)
"""

# ---------- Voix décrite par un texte (design) ----------
# Parler : locutrice nommée « Christine » (la voix française la mieux entraînée) pour une voix stable d'un essai à l'autre.
RUNNERS["parler"] = H + r"""
t = time.perf_counter()
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
repo = 'parler-tts/parler-tts-mini-multilingual-v1.1'
tok = AutoTokenizer.from_pretrained(repo)
desc = "Christine's voice is calm and clear, speaking at a moderate pace, with very clear audio quality and no background noise."
def load_m(dt):
    return ParlerTTSForConditionalGeneration.from_pretrained(repo, torch_dtype=dt).to(DEV)
def make_synth(m):
    dtok = AutoTokenizer.from_pretrained(m.config.text_encoder._name_or_path)
    d_in = dtok(desc, return_tensors='pt').to(DEV)
    def synth(txt):
        p_in = tok(txt, return_tensors='pt').to(DEV)
        with torch.no_grad():
            g = m.generate(input_ids=d_in.input_ids, attention_mask=d_in.attention_mask,
                           prompt_input_ids=p_in.input_ids, prompt_attention_mask=p_in.attention_mask)
        return g, m.config.sampling_rate
    return synth
model, synth, prec = load_with_half_fallback(load_m, make_synth)
load = time.perf_counter() - t
run(load, synth, info=prec + ', locutrice Christine')
"""

# Qwen3-TTS : un seul script pour les 3 usages (design, clonage, voix prédéfinies + instructions).
RUNNERS["qwen"] = H + r"""
t = time.perf_counter()
from qwen_tts import Qwen3TTSModel
REPO = os.environ['QWEN_REPO']; MODE = os.environ['QWEN_MODE']
INSTR = 'A clear, natural and warm female voice, speaking calmly in French.'
attn = ['sdpa']
def load_m(dt):
    kw = dict(device_map=('cuda:0' if DEV == 'cuda' else 'cpu'), dtype=dt)
    try:
        return Qwen3TTSModel.from_pretrained(REPO, attn_implementation='sdpa', **kw)
    except TypeError:
        attn[0] = 'défaut'
        return Qwen3TTSModel.from_pretrained(REPO, **kw)
note = []
def make_synth(m):
    if MODE == 'design':
        def synth(txt):
            w, sr = m.generate_voice_design(text=txt, language='French', instruct=INSTR); return w[0], sr
    elif MODE == 'clone':
        prompt = None
        try:   # voix de référence préparée UNE fois (hors chrono)
            prompt = m.create_voice_clone_prompt(ref_audio=REF, ref_text=REF_TEXT)
        except Exception as e:
            print('create_voice_clone_prompt indisponible :', e); note.append('référence retraitée à chaque appel')
        def synth(txt):
            if prompt is not None:
                w, sr = m.generate_voice_clone(text=txt, language='French', voice_clone_prompt=prompt)
            else:
                w, sr = m.generate_voice_clone(text=txt, language='French', ref_audio=REF, ref_text=REF_TEXT)
            return w[0], sr
    else:  # 'custom' : voix prédéfinie + instruction de style
        spk = None
        try:
            spk = list(m.get_supported_speakers())[0]
        except Exception as e:
            print('liste des voix indisponible :', e)
        def synth(txt):
            kw = dict(text=txt, language='French', instruct='Speak calmly and warmly.')
            if spk: kw['speaker'] = spk
            w, sr = m.generate_custom_voice(**kw); return w[0], sr
        if spk: note.append('voix ' + str(spk))
    return synth
model, synth, prec = load_with_half_fallback(load_m, make_synth)
load = time.perf_counter() - t
run(load, synth, info=', '.join([prec, 'attention ' + attn[0]] + note))
"""

# ---------- Clonage de voix (tous clonent la même voix « Claire ») ----------
# XTTS : la voix de référence est analysée une fois (latents), puis réutilisée.
RUNNERS["xtts"] = H + r"""
os.environ['COQUI_TOS_AGREED'] = '1'
t = time.perf_counter()
from TTS.api import TTS
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(DEV)
info = 'voix Claire préparée une fois'
try:
    tm = tts.synthesizer.tts_model
    gpt_lat, spk_emb = tm.get_conditioning_latents(audio_path=[REF])
    def synth(txt):
        with torch.no_grad():
            out = tm.inference(txt, 'fr', gpt_lat, spk_emb)
        return out['wav'], 24000
except Exception as e:
    print('latents indisponibles, API simple :', e); info = 'référence retraitée à chaque appel'
    def synth(txt):
        return np.array(tts.tts(text=txt, speaker_wav=REF, language='fr')), 24000
load = time.perf_counter() - t
run(load, synth, info=info)
"""

# F5-TTS français : on charge le fichier RÉDUIT (1,35 Go) et pas le gros (5,39 Go).
RUNNERS["f5"] = H + r"""
t = time.perf_counter()
from f5_tts.api import F5TTS
from huggingface_hub import list_repo_files, hf_hub_download
repo = 'RASPIAUDIO/F5-French-MixedSpeakers-reduced'
files = list_repo_files(repo)
ck = [f for f in files if 'reduced' in f and f.endswith(('.safetensors', '.pt'))] or [f for f in files if f.endswith(('.safetensors', '.pt'))]
ckpt = hf_hub_download(repo, ck[0])
vocab = [f for f in files if 'vocab' in f.lower()]
vfile = hf_hub_download(repo, vocab[0]) if vocab else ''
f5 = None
for arch in ['F5TTS_Base', 'F5TTS_v1_Base']:
    try:
        f5 = F5TTS(model=arch, ckpt_file=ckpt, vocab_file=vfile, device=DEV)
        f5.infer(ref_file=REF, ref_text=REF_TEXT, gen_text='Bonjour.')
        break
    except Exception as e:
        print('arch', arch, 'échec:', e); f5 = None
assert f5 is not None, 'F5-TTS: aucun checkpoint compatible'
load = time.perf_counter() - t
def synth(txt):
    wav, sr, _ = f5.infer(ref_file=REF, ref_text=REF_TEXT, gen_text=txt)
    return wav, sr
run(load, synth, info=ck[0] + ', architecture ' + arch + ', référence retraitée à chaque appel (pas d’option de cache)')
"""

# Chatterbox V3 : clone Claire ; la voix est préparée une fois (prepare_conditionals).
RUNNERS["chatterbox"] = H + r"""
t = time.perf_counter()
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
try:
    model = ChatterboxMultilingualTTS.from_pretrained(device=DEV, t3_model='v3'); version = 'V3'
except TypeError as e:
    print('t3_model non reconnu :', e)
    model = ChatterboxMultilingualTTS.from_pretrained(device=DEV); version = 'version par défaut (pas forcément V3)'
cached = False
try:
    model.prepare_conditionals(REF); cached = True
except Exception as e:
    print('prepare_conditionals indisponible :', e)
load = time.perf_counter() - t
def synth(txt):
    if cached:
        return model.generate(txt, language_id='fr'), model.sr
    return model.generate(txt, language_id='fr', audio_prompt_path=REF), model.sr
run(load, synth, info=version + (', voix Claire préparée une fois' if cached else ', référence retraitée à chaque appel') + ', filigrane PerTh inclus')
"""

RUNNERS["kokoclone"] = H + r"""
os.chdir('/content/kokoclone'); sys.path.insert(0, '/content/kokoclone')
t = time.perf_counter()
from core.cloner import KokoClone
cloner = KokoClone()
load = time.perf_counter() - t
def synth(txt):
    cloner.generate(text=txt, lang='fr', reference_audio=REF, output_path='/tmp/kc.wav')
    return (lambda: sf.read('/tmp/kc.wav')), None
run(load, synth, info='référence retraitée à chaque appel (pas d’option de cache), écriture fichier incluse')
"""

# OpenVoice V2 : sans l'option enable_watermark (qui plantait) ; filigrane désactivé pour ne pas fausser le temps.
RUNNERS["openvoice"] = H + r"""
from huggingface_hub import snapshot_download
t = time.perf_counter()
d = snapshot_download('myshell-ai/OpenVoiceV2')
from openvoice import se_extractor
from openvoice.api import ToneColorConverter
from melo.api import TTS
tcc = ToneColorConverter(d + '/converter/config.json', device=DEV)
tcc.watermark_model = None
tcc.load_ckpt(d + '/converter/checkpoint.pth')
target_se = None
for vad in (True, False):
    try:
        target_se, _ = se_extractor.get_se(REF, tcc, vad=vad); break
    except Exception as e:
        print('get_se vad=%s échec :' % vad, e)
assert target_se is not None, "OpenVoice : impossible d'extraire la voix de référence"
melo = TTS(language='FR', device=DEV)
spk = melo.hps.data.spk2id['FR']
source_se = torch.load(d + '/base_speakers/ses/fr.pth', map_location=DEV)
load = time.perf_counter() - t
def synth(txt):
    melo.tts_to_file(txt, spk, '/tmp/ov_src.wav', speed=1.0)
    tcc.convert(audio_src_path='/tmp/ov_src.wav', src_se=source_se, tgt_se=target_se, output_path='/tmp/ov_out.wav')
    return (lambda: sf.read('/tmp/ov_out.wav')), None
run(load, synth, info='MeloTTS + conversion de timbre, voix Claire préparée une fois, sans filigrane, fichiers intermédiaires inclus')
"""

# Zonos : banc de filtres du modèle de locuteur construit sur CPU puis déplacé sur GPU (corrige le plantage).
RUNNERS["zonos"] = H + r"""
import torchaudio
t = time.perf_counter()
from zonos.model import Zonos
from zonos.conditioning import make_cond_dict
import zonos.speaker_cloning as sc
_orig_fbank_init = sc.logFbankCal.__init__
def _fbank_init_cpu(self, *a, **k):
    with torch.device('cpu'):
        _orig_fbank_init(self, *a, **k)
sc.logFbankCal.__init__ = _fbank_init_cpu
def _move(o, depth=0):
    if isinstance(o, torch.nn.Module):
        o.to(DEV); return
    if depth < 3 and hasattr(o, '__dict__'):
        for v in list(vars(o).values()): _move(v, depth + 1)
model = Zonos.from_pretrained('Zyphra/Zonos-v0.1-transformer', device=DEV)
spk_model = sc.SpeakerEmbeddingLDA()
_move(spk_model)
model.spk_clone_model = spk_model
_a, sr0 = sf.read(REF, dtype='float32')
spk = model.make_speaker_embedding(torch.from_numpy(_a).unsqueeze(0), sr0)
load = time.perf_counter() - t
def synth(txt):
    cond = make_cond_dict(text=txt, speaker=spk, language='fr-fr')
    codes = model.generate(model.prepare_conditioning(cond))
    return model.autoencoder.decode(codes).cpu()[0], model.autoencoder.sampling_rate
run(load, synth, info='voix Claire préparée une fois')
"""

# CosyVoice 3 : voix de référence préparée une fois (add_zero_shot_spk).
RUNNERS["cosyvoice"] = H + r"""
sys.path.insert(0, '/content/CosyVoice'); sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')
os.chdir('/content/CosyVoice')
from huggingface_hub import snapshot_download
t = time.perf_counter()
snapshot_download('FunAudioLLM/Fun-CosyVoice3-0.5B-2512', local_dir='pretrained_models/Fun-CosyVoice3-0.5B')
from cosyvoice.cli.cosyvoice import AutoModel
cv = AutoModel(model_dir='pretrained_models/Fun-CosyVoice3-0.5B')
PROMPT = 'You are a helpful assistant.<|endofprompt|>' + REF_TEXT
cached = False
try:
    cached = bool(cv.add_zero_shot_spk(PROMPT, REF, 'claire'))
except Exception as e:
    print('add_zero_shot_spk indisponible :', e)
load = time.perf_counter() - t
def synth(txt):
    gen = (cv.inference_zero_shot(txt, '', '', zero_shot_spk_id='claire', stream=False) if cached
           else cv.inference_zero_shot(txt, PROMPT, REF, stream=False))
    return torch.cat([j['tts_speech'] for j in gen], dim=1), cv.sample_rate
run(load, synth, info='voix Claire préparée une fois' if cached else 'référence retraitée à chaque appel')
"""

for k, v in RUNNERS.items():
    open(f"/content/bench/run_{k}.py", "w", encoding="utf-8").write(v)
print("Étape 2 terminée ✅ :", len(RUNNERS), "scripts écrits")

In [ ]:
# ===== ÉTAPE 3 : installation de TOUS les modèles en parallèle (≈ 15-30 min) =====
# Ce n'est PAS chronométré. Les échecs sont normaux : on les note.
import subprocess, os, time
from concurrent.futures import ThreadPoolExecutor
os.environ["UV_CACHE_DIR"] = "/content/uvcache"
ENVS = "/content/envs"

MODELS = [
 dict(name="kokoro", py="3.11", pkgs="kokoro soundfile", hf=["hexgrad/Kokoro-82M"]),
 dict(name="melo", py="3.10", pkgs="git+https://github.com/myshell-ai/MeloTTS.git soundfile",
      post=["{py} -m unidic download", "{py} -c \"import nltk; [nltk.download(x) for x in ['averaged_perceptron_tagger','averaged_perceptron_tagger_eng','cmudict']]\""],
      hf=["myshell-ai/MeloTTS-French"]),
 dict(name="kokoclone", py="3.12", pkgs="soundfile",
      pre=["rm -rf /content/kokoclone; git clone https://github.com/Ashish-Patnaik/kokoclone.git /content/kokoclone"],
      post=["uv pip install --python {py} -r /content/kokoclone/requirements.txt",
            "uv pip uninstall --python {py} onnxruntime; uv pip install --python {py} onnxruntime-gpu"]),
 dict(name="parler", py="3.11", pkgs="git+https://github.com/huggingface/parler-tts.git soundfile",
      post=["uv pip install --python {py} sentencepiece==0.2.0"], hf=["parler-tts/parler-tts-mini-multilingual-v1.1"]),
 dict(name="xtts", py="3.11", pkgs="coqui-tts torch==2.6.0 torchaudio==2.6.0 'transformers<5' 'numpy<2.1' soundfile"),
 dict(name="f5", py="3.11", pkgs="f5-tts soundfile",
      hf=[("RASPIAUDIO/F5-French-MixedSpeakers-reduced", ["*reduced*", "vocab*"])]),
 dict(name="qwen", py="3.12", pkgs="qwen-tts soundfile",
      hf=["Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
          "Qwen/Qwen3-TTS-12Hz-0.6B-Base", "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"]),
 dict(name="chatterbox", py="3.11", pkgs="chatterbox-tts soundfile", post=["uv pip install --python {py} -U chatterbox-tts"]),
 dict(name="openvoice", py="3.10", pkgs="git+https://github.com/myshell-ai/MeloTTS.git soundfile",
      post=["uv pip install --python {py} --no-deps git+https://github.com/myshell-ai/OpenVoice.git",
            "uv pip install --python {py} wavmark 'setuptools<80' faster-whisper whisper-timestamped pydub",
            "{py} -m unidic download",
            "{py} -c \"import nltk; [nltk.download(x) for x in ['averaged_perceptron_tagger','averaged_perceptron_tagger_eng','cmudict']]\""],
      hf=["myshell-ai/OpenVoiceV2", "myshell-ai/MeloTTS-French"]),
 dict(name="zonos", py="3.11", pkgs="soundfile",
      pre=["rm -rf /content/Zonos; git clone https://github.com/Zyphra/Zonos.git /content/Zonos"],
      post=["uv pip install --python {py} -e /content/Zonos"], hf=["Zyphra/Zonos-v0.1-transformer"]),
 dict(name="cosyvoice", py="3.10", pkgs="pynini onnxruntime wheel",
      pre=["rm -rf /content/CosyVoice; git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice",
           "sed -E -i '/tensorrt|deepspeed|vllm|onnxruntime-gpu|extra-index|^-f /d' /content/CosyVoice/requirements.txt"],
      post=["uv pip install --python {py} --no-build-isolation openai-whisper==20231117",
            "uv pip install --python {py} -r /content/CosyVoice/requirements.txt"]),
]

def sh(cmd, logf, timeout=1500):
    logf.write(f"\n$ {cmd}\n"); logf.flush()
    r = subprocess.run(cmd, shell=True, stdout=logf, stderr=subprocess.STDOUT, timeout=timeout)
    return r.returncode == 0

def install(m):
    env = f"{ENVS}/{m['name']}"; py = f"{env}/bin/python"; t0 = time.time()
    with open(f"/content/bench/logs/install_{m['name']}.log", "w") as lg:
        try:
            ok = sh(f"uv venv {env} --python {m['py']}", lg)
            ok = sh(f"uv pip install --python {py} 'setuptools<81' wheel six", lg) and ok
            for c in m.get("pre", []):
                ok = sh(c.format(py=py, env=env), lg) and ok
            ok = ok and sh(f"uv pip install --python {py} {m['pkgs']}", lg)
            for c in m.get("post", []):
                if ok: ok = sh(c.format(py=py, env=env), lg)
            sh(f"uv pip install --python {py} 'setuptools<81' six", lg)
        except Exception as e:
            lg.write(f"EXCEPTION {e}\n"); ok = False
    m["install_ok"] = ok
    print(("✅" if ok else "❌"), "install", m["name"], f"({time.time()-t0:.0f}s)", flush=True)

def prefetch(item):
    from huggingface_hub import snapshot_download
    repo, patterns = item if isinstance(item, tuple) else (item, None)
    try:
        snapshot_download(repo, allow_patterns=patterns)
        print("⬇️  poids téléchargés :", repo, flush=True)
    except Exception as e:
        print("⚠️  prefetch échoué", repo, str(e)[:120], flush=True)

t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as ex:
    futs = [ex.submit(install, m) for m in MODELS] + [ex.submit(prefetch, r) for m in MODELS for r in m.get("hf", [])]
    for f in futs: f.result()
print(f"\nÉtape 3 terminée en {(time.time()-t0)/60:.1f} min ✅")
print("Échecs d'installation :", [m['name'] for m in MODELS if not m.get('install_ok')] or "aucun")

In [ ]:
# ===== ÉTAPE 4 : chronométrage, un modèle à la fois (≈ 40-70 min) =====
import subprocess, os, json
import pandas as pd

BY_ENV = {m["name"]: m for m in MODELS}
# Chaque ligne = un modèle (ou une variante) : script + environnement + réglages.
# Kokoro passe en premier : il fabrique la voix de référence « Claire » utilisée par tous les modèles de clonage.
RUNS = [
 dict(name="kokoro",       env="kokoro",     cat="voix prédéfinie", label="Kokoro-82M (hexgrad/Kokoro-82M v1.0)"),
 dict(name="melo",         env="melo",       cat="voix prédéfinie", label="MeloTTS (myshell-ai/MeloTTS-French)"),
 dict(name="qwen_custom",  env="qwen",       cat="voix prédéfinie + instructions", label="Qwen3-TTS 1.7B-CustomVoice (demi-précision)",
      QWEN_REPO="Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice", QWEN_MODE="custom", HALF="1"),
 dict(name="parler",       env="parler",     cat="design (description)", label="Parler-TTS Mini Multilingual v1.1 (float32)"),
 dict(name="parler_half",  env="parler",     cat="design (description)", label="Parler-TTS Mini Multilingual v1.1 (demi-précision)", HALF="1"),
 dict(name="qwen_design",  env="qwen",       cat="design (description)", label="Qwen3-TTS 1.7B-VoiceDesign (float32)",
      QWEN_REPO="Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", QWEN_MODE="design"),
 dict(name="qwen_design_half", env="qwen",   cat="design (description)", label="Qwen3-TTS 1.7B-VoiceDesign (demi-précision)",
      QWEN_REPO="Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", QWEN_MODE="design", HALF="1"),
 dict(name="qwen_base17",  env="qwen",       cat="clonage", label="Qwen3-TTS 1.7B-Base (demi-précision)",
      QWEN_REPO="Qwen/Qwen3-TTS-12Hz-1.7B-Base", QWEN_MODE="clone", HALF="1"),
 dict(name="qwen_base06",  env="qwen",       cat="clonage", label="Qwen3-TTS 0.6B-Base (demi-précision)",
      QWEN_REPO="Qwen/Qwen3-TTS-12Hz-0.6B-Base", QWEN_MODE="clone", HALF="1"),
 dict(name="kokoclone",    env="kokoclone",  cat="clonage", label="KokoClone (Kokoro-ONNX + Kanade)", ONNX_PROVIDER="CUDAExecutionProvider"),
 dict(name="xtts",         env="xtts",       cat="clonage", label="XTTS-v2 (coqui-tts, fork Idiap)"),
 dict(name="f5",           env="f5",         cat="clonage", label="F5-TTS (RASPIAUDIO/F5-French-MixedSpeakers-reduced)"),
 dict(name="chatterbox",   env="chatterbox", cat="clonage", label="Chatterbox Multilingual V3"),
 dict(name="openvoice",    env="openvoice",  cat="clonage", label="OpenVoice V2"),
 dict(name="zonos",        env="zonos",      cat="clonage", label="Zonos-v0.1-transformer"),
 dict(name="cosyvoice",    env="cosyvoice",  cat="clonage", label="CosyVoice 3 (Fun-CosyVoice3-0.5B-2512)"),
]
RUN_BY_NAME = {r["name"]: r for r in RUNS}
RUN_TIMEOUT = 1800   # 30 min max par modèle
EXTRA_ENV = ("HALF", "ONNX_PROVIDER", "QWEN_REPO", "QWEN_MODE")
rows = []

def run_one(name):
    r = RUN_BY_NAME[name]; m = BY_ENV[r["env"]]
    row = dict(clé=name, modèle=r["label"], catégorie=r["cat"], statut="", temps=None, moyenne_s=None, écart_type_s=None,
               durées_audio_s=None, vram_Go=None, charge_s=None, torch_device="", onnx="", eos_forcé=None, détail="", erreur="")
    if not m.get("install_ok"):
        row.update(statut="❌ installation échouée", erreur="voir logs/install_%s.log" % r["env"]); return row
    env = dict(os.environ, BENCH_NAME=name, N_RUNS=str(N_RUNS), PYTHONUNBUFFERED="1", COQUI_TOS_AGREED="1",
               TOKENIZERS_PARALLELISM="false", MPLBACKEND="Agg")
    for k in EXTRA_ENV:
        if k in r: env[k] = r[k]
    try:
        p = subprocess.run([f"{ENVS}/{r['env']}/bin/python", f"/content/bench/run_{r['env']}.py"],
                           env=env, capture_output=True, text=True, timeout=RUN_TIMEOUT)
        out = p.stdout + "\n" + p.stderr
        open(f"/content/bench/logs/run_{name}.log", "w").write(out)
        line = [l for l in p.stdout.splitlines() if l.startswith("RESULT ")]
        if line:
            d = json.loads(line[-1][7:])
            row.update(statut="✅ OK", temps=d["times"], moyenne_s=d["mean_s"], écart_type_s=d["std_s"], durées_audio_s=d["durs"],
                       vram_Go=d["vram_gb"], charge_s=d["load_s"], torch_device=d["device"], onnx=",".join(d["ort"]), détail=d["info"])
            if r["env"] == "chatterbox":
                row["eos_forcé"] = out.count("forcing EOS")
        else:
            row.update(statut="❌ erreur à l'exécution", erreur=" | ".join(out.strip().splitlines()[-2:])[:200])
    except subprocess.TimeoutExpired:
        row.update(statut="⏱️ trop lent (>30 min)", erreur="timeout")
    return row

for r in RUNS:
    print(f"▶️  {r['label']} ...", flush=True)
    row = run_one(r["name"]); rows.append(row)
    print("   →", row["statut"], "| temps (s) =", row["temps"], "| moyenne =", row["moyenne_s"], "s |", row["détail"] or row["erreur"], flush=True)
    pd.DataFrame(rows).to_csv("/content/bench/logs/details_techniques.csv", index=False)

df = pd.DataFrame(rows)
print("\n===== TEMPS D'INFÉRENCE (trié par moyenne) =====")
display(df.sort_values("moyenne_s", na_position="last")[["modèle", "catégorie", "statut", "temps", "moyenne_s"]].reset_index(drop=True))

In [ ]:
# ===== ÉTAPE 5 : contrôle : Whisper vérifie que chaque audio dit bien la phrase (≈ 5-10 min) =====
# Sert à repérer les « faux » essais rapides (phrase coupée, répétée, incompréhensible).
import glob, re, unicodedata, os
import numpy as np, soundfile as sf, pandas as pd, torch
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device=0 if torch.cuda.is_available() else -1)

def words(s):
    s = unicodedata.normalize("NFC", s.lower()).replace("’", "'")
    s = re.sub(r"[^\w\s']", " ", s).replace("'", "' ")
    return s.split()

def wer(ref, hyp):
    r, h = words(ref), words(hyp)
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
            prev, d[j] = d[j], cur
    return d[len(h)] / max(len(r), 1)

qrows = []
for f in sorted(glob.glob("/content/bench/out/*_r*.wav")):
    key, run_i = re.match(r"(.+)_r(\d+)\.wav$", os.path.basename(f)).groups()
    a, sr = sf.read(f, dtype="float32")
    if a.ndim > 1: a = a.mean(axis=1)
    hyp = asr({"raw": a, "sampling_rate": sr}, generate_kwargs={"language": "french", "task": "transcribe"})["text"]
    qrows.append(dict(clé=key, essai=int(run_i), wer=round(wer(TEXT, hyp), 3), transcription=hyp.strip()))
qdf = pd.DataFrame(qrows, columns=["clé", "essai", "wer", "transcription"])
qdf.to_csv("/content/bench/logs/transcriptions.csv", index=False)
print("Contrôle terminé ✅ :", len(qdf), "audios transcrits")

In [ ]:
# ===== ÉTAPE 6 : tableau final (temps d'inférence + remarque), écoute, téléchargement =====
from IPython.display import Audio, display
import shutil
WER_MAX = 0.15   # au-delà de 15 % de mots faux, l'essai est signalé

def remarque(r):
    if not str(r["statut"]).startswith("✅"):
        return "ne tourne pas : " + (str(r["erreur"]) or str(r["statut"]))
    notes = []
    q = qdf[qdf["clé"] == r["clé"]].sort_values("essai")
    bad = q[q["wer"] > WER_MAX]
    if len(bad):
        notes.append("audio incorrect aux essais %s (jusqu'à %d %% de mots faux, voir transcriptions.csv)"
                     % (", ".join(str(x) for x in bad["essai"]), round(100 * bad["wer"].max())))
    eos = r.get("eos_forcé")
    if pd.notna(eos) and eos > 0:
        notes.append("fin de phrase forcée par le modèle %d fois" % int(eos))
    t = r["temps"] if isinstance(r["temps"], list) else []
    if t and max(t) > 1.5 * min(t):
        notes.append("temps instable (%.2f à %.2f s)" % (min(t), max(t)))
    if isinstance(r["détail"], str) and r["détail"]:
        notes.append(r["détail"])
    return " ; ".join(notes)

MOY = "moyenne des %d essais (s)" % N_RUNS
final = pd.DataFrame({"catégorie": df["catégorie"], "modèle": df["modèle"]})
for i in range(N_RUNS):
    final["essai %d (s)" % (i + 1)] = df["temps"].map(lambda t, i=i: t[i] if isinstance(t, list) and len(t) > i else None)
final[MOY] = df["moyenne_s"]
final["remarque"] = df.apply(remarque, axis=1)

# La comparaison se fait PAR CATÉGORIE : les modèles d'une même catégorie reçoivent la même chose en entrée.
CATS = [
 ("clonage", "CLONAGE DE VOIX : chaque modèle reçoit le texte + le même enregistrement de référence (« Claire »)"),
 ("design (description)", "DESIGN DE VOIX : chaque modèle reçoit le texte + une description écrite de la voix (pas d'enregistrement)"),
 ("voix prédéfinie + instructions", "VOIX PRÉDÉFINIE + INSTRUCTIONS : le texte + une voix intégrée au modèle + une consigne de style"),
 ("voix prédéfinie", "VOIX PRÉDÉFINIE (référence de vitesse) : le texte + une voix intégrée au modèle, ni clonage ni design"),
]
# Rang seulement pour les modèles dont TOUS les essais sont corrects (sinon un temps « rapide » mais faux serait récompensé).
bad_keys = set(qdf.loc[qdf["wer"] > WER_MAX, "clé"])
valid = df["statut"].astype(str).str.startswith("✅") & ~df["clé"].isin(bad_keys)
final["_moy_valide"] = final[MOY].where(valid)
final["rang dans la catégorie"] = final.groupby("catégorie")["_moy_valide"].rank(method="min").astype("Int64")
final = final.drop(columns="_moy_valide")
order = {c: k for k, (c, _) in enumerate(CATS)}
final["_o"] = final["catégorie"].map(order)
final = final.sort_values(["_o", MOY], na_position="last").drop(columns="_o").reset_index(drop=True)
cols = ["catégorie", "rang dans la catégorie", "modèle"] + ["essai %d (s)" % (i + 1) for i in range(N_RUNS)] + [MOY, "remarque"]
final = final[cols]
final.to_csv("/content/bench/resultats_inference.csv", index=False)

pd.set_option("display.max_colwidth", 200)
print("===== TEMPS D'INFÉRENCE (GPU : %s) =====" % GPU_NAME)
print("⚠️  La comparaison se fait PAR CATÉGORIE : les catégories ne reçoivent pas les mêmes entrées et ne font pas le même travail.")
print("    Le rang (1 = le plus rapide) n'est calculé qu'entre modèles d'une même catégorie, et seulement si ses 5 audios sont corrects (sinon : pas de rang, voir la remarque).\n")
for cat, titre in CATS:
    part = final[final["catégorie"] == cat].drop(columns="catégorie")
    if len(part):
        print("▶ " + titre)
        display(part.reset_index(drop=True))
        print()

for w in sorted(glob.glob("/content/bench/out/*.wav")):
    if re.search(r"_r\d+\.wav$", w): continue      # dernier essai de chaque modèle seulement
    print(os.path.basename(w)); display(Audio(w))

shutil.copy("/content/bench/ref_fr.wav", "/content/bench/out/_reference_claire.wav")
shutil.make_archive("/content/bench_audio", "zip", "/content/bench/out")
shutil.make_archive("/content/bench_logs", "zip", "/content/bench/logs")
from google.colab import files
files.download("/content/bench/resultats_inference.csv")
files.download("/content/bench_audio.zip")
files.download("/content/bench_logs.zip")

In [ ]:
# (Optionnel) Relancer UN seul modèle, par ex. "zonos", puis refaire les étapes 5 et 6 :
# rows = [r for r in rows if r["clé"] != "zonos"] + [run_one("zonos")]; df = pd.DataFrame(rows)
# Pour voir pourquoi ça a échoué :  !tail -n 30 /content/bench/logs/run_zonos.log   ou   install_zonos.log